In [ ]:

FREESOLV_CSV = '/kaggle/input/datasets/kaalmurlidhar/esolandfreesol/freesolv.csv'
ESOL_CSV = '/kaggle/input/datasets/kaalmurlidhar/esolandfreesol/delaney-processed.csv'


# -- freesolv settings 
FREESOLV_EPOCHS         = 200
FREESOLV_LR             = 5e-5
FREESOLV_BS             = 16
FREESOLV_WEIGHT_DECAY   = 1e-5
FREESOLV_SEED           = 2025
FREESOLV_EARLY_STOP     = 30
FREESOLV_LR_FACTOR      = 0.7
FREESOLV_LR_PATIENCE    = 5
FREESOLV_LR_MIN         = 1e-6
FREESOLV_PREDICTOR_DROP = 0.2   # predictor uses BatchNorm + Dropout

# -- esol settings
ESOL_EPOCHS         = 100
ESOL_LR             = 2e-5
ESOL_BS             = 16
ESOL_WEIGHT_DECAY   = 0.0      # original does not set weight_decay on optimizer
ESOL_SEED           = 2024
ESOL_EARLY_STOP     = 50
ESOL_LR_FACTOR      = 0.6
ESOL_LR_PATIENCE    = 10
ESOL_LR_MIN         = 5e-6
ESOL_PREDICTOR_DROP = 0.1      # predictor uses only Dropout, no BatchNorm

# -- shared model architecture (identical in both original scripts)
D_MODEL    = 768
N_LAYERS   = 6
N_HEADS    = 12
D_K        = 64
D_V        = 64
D_FF       = 768 * 4   # = 3072
VOCAB_SIZE = 83
MAXLEN     = 501

# -- gcn architecture
# freesolv: GCN(num_layer=5, emb_dim=300, feat_dim=300, drop_ratio=0.15)
# esol:     GCN(num_layer=5, emb_dim=300, feat_dim=300, drop_ratio=0.1)
GCN_LAYERS   = 5
GCN_EMB_DIM  = 300
GCN_FEAT_DIM = 300

# combined dimension entering the predictor.
# d_model (768) + gcn feat_dim (300) = 1068. same in both scripts.
COMBINED_DIM = D_MODEL + GCN_FEAT_DIM  # 1068

# ============================================================
# section 1: dependency installation
# ============================================================

import subprocess
import sys
import torch as _torch_check


def _install_pyg_wheels():
    
    torch_version = _torch_check.__version__.split('+')[0]  
    if _torch_check.cuda.is_available():
        cuda_str = 'cu' + _torch_check.version.cuda.replace('.', '')
    else:
        cuda_str = 'cpu'

    wheel_url = f'https://data.pyg.org/whl/torch-{torch_version}+{cuda_str}.html'
    print(f'  using pyg wheel index: {wheel_url}')

    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'torch_scatter', 'torch_sparse',
        '-f', wheel_url
    ])

def _install_if_missing(import_name, pip_name, fallback_pip_name=None):
    
    try:
        __import__(import_name)
        print(f'  {import_name}: already installed, skipping.')
    except ImportError:
        print(f'  installing {pip_name}...')
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])
        except subprocess.CalledProcessError:
            if fallback_pip_name:
                print(f'  {pip_name} failed, trying fallback {fallback_pip_name}...')
                subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', fallback_pip_name])
            else:
                raise

print('installing dependencies...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'])
_install_pyg_wheels()   # pre-built wheels for torch_scatter and torch_sparse
_install_if_missing('rdkit', 'rdkit', fallback_pip_name='rdkit-pypi')
_install_if_missing('nltk',  'nltk')
_install_if_missing('six',   'six')

import nltk
nltk.download('punkt', quiet=True)
print('all dependencies installed.')


import os
import sys
import copy
import math
import time
import contextlib
import re as _re
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, Subset

from rdkit import Chem
from rdkit.Chem.rdchem import BondType as BT
from rdkit import RDLogger

import six

RDLogger.DisableLog('rdApp.*')

from torch_geometric.nn import global_mean_pool
from torch_geometric.utils import add_self_loops
from torch_scatter import scatter_add

# zinc cfg grammar and token sequence tools

# reproduced from preprocess/grammar.py and preprocess/parse_trees.py.

GRAM_STRING = """smiles -> chain
atom -> bracket_atom
atom -> aliphatic_organic
atom -> aromatic_organic
aliphatic_organic -> 'B'
aliphatic_organic -> 'C'
aliphatic_organic -> 'N'
aliphatic_organic -> 'O'
aliphatic_organic -> 'S'
aliphatic_organic -> 'P'
aliphatic_organic -> 'F'
aliphatic_organic -> 'I'
aliphatic_organic -> 'Cl'
aliphatic_organic -> 'Br'
aromatic_organic -> 'c'
aromatic_organic -> 'n'
aromatic_organic -> 'o'
aromatic_organic -> 's'
bracket_atom -> '[' BAI ']'
BAI -> isotope symbol BAC
BAI -> symbol BAC
BAI -> isotope symbol
BAI -> symbol
BAC -> chiral BAH
BAC -> BAH
BAC -> chiral
BAH -> hcount BACH
BAH -> BACH
BAH -> hcount
BACH -> charge class
BACH -> charge
BACH -> class
symbol -> aliphatic_organic
symbol -> aromatic_organic
isotope -> DIGIT
isotope -> DIGIT DIGIT
isotope -> DIGIT DIGIT DIGIT
DIGIT -> '1'
DIGIT -> '2'
DIGIT -> '3'
DIGIT -> '4'
DIGIT -> '5'
DIGIT -> '6'
DIGIT -> '7'
DIGIT -> '8'
chiral -> '@'
chiral -> '@@'
hcount -> 'H'
hcount -> 'H' DIGIT
charge -> '-'
charge -> '-' DIGIT
charge -> '-' DIGIT DIGIT
charge -> '+'
charge -> '+' DIGIT
charge -> '+' DIGIT DIGIT
bond -> '-'
bond -> '='
bond -> '#'
bond -> '/'
bond -> '\\\\'
bond -> '.'
ringbond -> DIGIT
ringbond -> bond DIGIT
branched_atom -> atom
branched_atom -> atom RB
branched_atom -> atom BB
branched_atom -> atom RB BB
RB -> RB ringbond
RB -> ringbond
BB -> BB branch
BB -> branch
branch -> '(' chain ')'
branch -> '(' bond chain ')'
chain -> branched_atom
chain -> chain branched_atom
chain -> chain bond branched_atom
symbol -> element_symbols
aromatic_organic -> 'p'
element_symbols -> 'H'
class -> DIGIT
Nothing -> None"""

GCFG = nltk.CFG.fromstring(GRAM_STRING)


_RULE_LOOKUP = {}
for _idx, _prod in enumerate(GCFG.productions()):
    _lhs = _prod.lhs().symbol()
    _rhs = tuple(
        s.symbol() if not isinstance(s, str) else s
        for s in _prod.rhs()
    )
    _RULE_LOOKUP[(_lhs, _rhs)] = _idx + 1  # 1-based

# create the parser once. creating ChartParser is expensive; reuse one instance.
_PARSER = nltk.ChartParser(GCFG)

# smiles tokenizer regex. handles multi-character tokens like Cl, Br, [nH], @@.
_SMILES_RE = _re.compile(r'(\[[^\]]+\]|Br|Cl|@@|[BCNOPSFIbcnosp=#@+\-\/\\\.1-9])')

# grammar token integers: values 1-80 are real grammar rule tokens.
_GRAMMAR_TOKEN_SET = set(range(1, 81))


def _tokenize(smiles):
    return _SMILES_RE.findall(smiles)


def encode_smiles(smiles):
    # encode a smiles string into a list of 1-based grammar rule integers.
    # raises ValueError if the smiles cannot be parsed by the zinc cfg.
    tokens = _tokenize(smiles)
    if not tokens:
        raise ValueError(f'tokenizer returned empty list: {smiles}')
    trees = list(_PARSER.parse(tokens))
    if not trees:
        raise ValueError(f'cfg parse failed: {smiles}')
    tree = trees[0]
    rule_indices = []
    for subtree in tree.subtrees():
        if subtree.height() > 1:
            lhs = subtree.label()
            rhs = tuple(
                child.label() if isinstance(child, nltk.Tree) else child
                for child in subtree
            )
            idx = _RULE_LOOKUP.get((lhs, rhs))
            if idx is not None:
                rule_indices.append(idx)
    return rule_indices


def construct_token_sequence(rule_index_list, max_len=500):
   

    if len(rule_index_list) > max_len:
        rule_index_list = rule_index_list[:max_len]

    padding_list = ['[PAD]'] * (max_len - len(rule_index_list))
    tokens = ['[GLO]'] + rule_index_list + padding_list

    tokens_idx = []
    atom_mask  = []

    for token in tokens:
        if token in _GRAMMAR_TOKEN_SET:
            atom_mask.append(1)
            tokens_idx.append(token)
        elif token == '[GLO]':
            atom_mask.append(1)
            tokens_idx.append(82)
        else:
            atom_mask.append(0)
            tokens_idx.append(0)

    return tokens_idx, atom_mask


#electron-relevant token set

# reproduced from preprocess/property_tags.py.


ELECTRON_RELEVANT_TOKEN_SET = {7, 8, 9, 15, 16, 17, 18, 57, 58, 78}

#model architecture

# the Embedding class includes the property-aware electron_bias modification.
# all other components (MultiHeadAttention, EncoderLayer, GCN, CombinedModel)
# are reproduced exactly from the original repository.
.

def get_attn_pad_mask(seq_q):
    batch_size, seq_len = seq_q.size()
    pad_attn_mask = seq_q.data.eq(0).unsqueeze(1)
    return pad_attn_mask.expand(batch_size, seq_len, seq_len)


class Embedding(nn.Module):
    # property-aware embedding.
    # adds a learnable scalar bias (electron_bias) to the embedding vectors
    # of electron-relevant grammar rule tokens before layernorm.
    # formula: h_0 = LayerNorm(E_tok(x) + E_pos(p) + electron_mask(x) * beta)

    def __init__(self, vocab_size, d_model, maxlen):
        super().__init__()
        self.tok_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(maxlen, d_model)
        self.norm      = nn.LayerNorm(d_model)

        # single learnable scalar bias. initialized to 0.1.
        # updated by the optimizer during training.
        self.electron_bias = nn.Parameter(torch.tensor(0.1))

        # fixed buffer of electron-relevant token integers.
        # moves to the correct device with model.to(device).
        self.register_buffer(
            'electron_token_indices',
            torch.tensor(sorted(ELECTRON_RELEVANT_TOKEN_SET), dtype=torch.long)
        )

    def forward(self, x):
        # x: [batch, seq_len] integer token indices.
        seq_len = x.size(1)
        pos = torch.arange(seq_len, dtype=torch.long).unsqueeze(0).expand_as(x).to(x.device)

        tok_embedding = self.tok_embed(x)
        pos_embedding = self.pos_embed(pos)

        # build binary electron-relevance mask.
        # broadcasts [batch, seq_len, 1] vs [num_electron_tokens] -> [batch, seq_len, num_electron_tokens]
        # .any(dim=-1) -> [batch, seq_len], then unsqueeze for d_model broadcast.
        electron_mask = (x.unsqueeze(-1) == self.electron_token_indices).any(dim=-1)
        electron_mask = electron_mask.unsqueeze(-1).float()

        electron_bump = electron_mask * self.electron_bias

        return self.norm(tok_embedding + pos_embedding + electron_bump)


class ScaledDotProductAttention(nn.Module):
    def __init__(self, d_k):
        super().__init__()
        self.d_k = d_k

    def forward(self, Q, K, V, attn_mask):
        scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(self.d_k)
        scores.masked_fill_(attn_mask, -1e9)
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(attn, V)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, d_k, d_v, n_heads):
        super().__init__()
        self.d_k = d_k
        self.d_v = d_v
        self.n_heads = n_heads
        self.W_Q = nn.Linear(d_model, d_k * n_heads)
        self.W_K = nn.Linear(d_model, d_k * n_heads)
        self.W_V = nn.Linear(d_model, d_v * n_heads)
        self.linear    = nn.Linear(n_heads * d_v, d_model)
        self.layernorm = nn.LayerNorm(d_model)

    def forward(self, Q, K, V, attn_mask):
        residual, batch_size = Q, Q.size(0)
        q_s = self.W_Q(Q).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        k_s = self.W_K(K).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        v_s = self.W_V(V).view(batch_size, -1, self.n_heads, self.d_v).transpose(1, 2)
        attn_mask = attn_mask.unsqueeze(1).repeat(1, self.n_heads, 1, 1)
        context = ScaledDotProductAttention(self.d_k)(q_s, k_s, v_s, attn_mask)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.n_heads * self.d_v)
        return self.layernorm(self.linear(context) + residual)


class PoswiseFeedForwardNet(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(d_model, d_ff, bias=False),
            nn.ReLU(),
            nn.Linear(d_ff, d_model, bias=False),
        )
        self.layernorm = nn.LayerNorm(d_model)

    def forward(self, inputs):
        return self.layernorm(self.fc(inputs) + inputs)


class EncoderLayer(nn.Module):
    def __init__(self, d_model, d_k, d_v, n_heads, d_ff):
        super().__init__()
        self.enc_self_attn = MultiHeadAttention(d_model, d_k, d_v, n_heads)
        self.pos_ffn       = PoswiseFeedForwardNet(d_model, d_ff)

    def forward(self, enc_inputs, enc_self_attn_mask):
        enc_outputs = self.enc_self_attn(enc_inputs, enc_inputs, enc_inputs, enc_self_attn_mask)
        return self.pos_ffn(enc_outputs)


class BERT_atom_embedding_generator(nn.Module):
    # property-aware transformer encoder.
    # identical interface to the original BERT_atom_embedding_generator
    # in model/my_nn.py. the only change is the Embedding class used inside.

    def __init__(self, d_model, n_layers, vocab_size, maxlen, d_k, d_v, n_heads, d_ff,
                 global_label_dim, atom_label_dim):
        super().__init__()
        self.d_model   = d_model
        self.embedding = Embedding(vocab_size, d_model, maxlen)
        self.layers    = nn.ModuleList([
            EncoderLayer(d_model, d_k, d_v, n_heads, d_ff) for _ in range(n_layers)
        ])

    def forward(self, input_ids):
        output    = self.embedding(input_ids)
        attn_mask = get_attn_pad_mask(input_ids)
        for layer in self.layers:
            output = layer(output, attn_mask)
        return output[:, 0, :]  # [batch, d_model] — [GLO] token


# gcn components. reproduced from model/gcn_finetune.py.
_NUM_ATOM_TYPE      = 119
_NUM_CHIRALITY_TAG  = 3
_NUM_BOND_TYPE      = 5
_NUM_BOND_DIRECTION = 3


def _gcn_norm(edge_index, num_nodes=None):
    from torch_geometric.utils.num_nodes import maybe_num_nodes
    num_nodes   = maybe_num_nodes(edge_index, num_nodes)
    edge_weight = torch.ones(edge_index.size(1), device=edge_index.device)
    row, col    = edge_index[0], edge_index[1]
    deg         = scatter_add(edge_weight, col, dim=0, dim_size=num_nodes)
    deg_inv_sqrt = deg.pow_(-0.5)
    deg_inv_sqrt.masked_fill_(deg_inv_sqrt == float('inf'), 0)
    return edge_index, deg_inv_sqrt[row] * edge_weight * deg_inv_sqrt[col]


from torch_geometric.nn import MessagePassing
import torch_sparse


class GCNConv(MessagePassing):
    def __init__(self, emb_dim, aggr='add'):
        super().__init__()
        self.emb_dim = emb_dim
        self.aggr    = aggr
        self.weight  = nn.Parameter(torch.Tensor(emb_dim, emb_dim))
        self.bias    = nn.Parameter(torch.Tensor(emb_dim))
        stdv = math.sqrt(6.0 / (emb_dim + emb_dim))
        self.weight.data.uniform_(-stdv, stdv)
        self.bias.data.fill_(0)
        self.edge_embedding1 = nn.Embedding(_NUM_BOND_TYPE, 1)
        self.edge_embedding2 = nn.Embedding(_NUM_BOND_DIRECTION, 1)
        nn.init.xavier_uniform_(self.edge_embedding1.weight.data)
        nn.init.xavier_uniform_(self.edge_embedding2.weight.data)

    def forward(self, x, edge_index, edge_attr):
        edge_index = add_self_loops(edge_index, num_nodes=x.size(0))[0]
        self_loop_attr = torch.zeros(x.size(0), 2, device=edge_attr.device, dtype=edge_attr.dtype)
        self_loop_attr[:, 0] = 4
        edge_attr = torch.cat((edge_attr, self_loop_attr), dim=0)
        edge_embeddings = self.edge_embedding1(edge_attr[:, 0]) + self.edge_embedding2(edge_attr[:, 1])
        edge_index, _ = _gcn_norm(edge_index)
        x = x @ self.weight
        out = self.propagate(edge_index, x=x, edge_attr=edge_embeddings, size=None)
        return out + self.bias

    def message(self, x_j, edge_attr):
        return x_j if edge_attr is None else edge_attr + x_j

    def message_and_aggregate(self, adj_t, x):
        return torch_sparse.matmul(adj_t, x, reduce=self.aggr)


class GCN(nn.Module):
    def __init__(self, num_layer=5, emb_dim=300, feat_dim=300, drop_ratio=0.15, pool='mean'):
        super().__init__()
        if num_layer < 2:
            raise ValueError('gcn requires at least 2 layers.')
        self.num_layer  = num_layer
        self.drop_ratio = drop_ratio
        self.x_embedding1 = nn.Embedding(_NUM_ATOM_TYPE,     emb_dim)
        self.x_embedding2 = nn.Embedding(_NUM_CHIRALITY_TAG, emb_dim)
        nn.init.xavier_uniform_(self.x_embedding1.weight.data)
        nn.init.xavier_uniform_(self.x_embedding2.weight.data)
        self.gnns        = nn.ModuleList([GCNConv(emb_dim) for _ in range(num_layer)])
        self.batch_norms = nn.ModuleList([nn.BatchNorm1d(emb_dim) for _ in range(num_layer)])
        self.pool        = global_mean_pool
        self.feat_lin    = nn.Linear(emb_dim, feat_dim)

    def forward(self, x, edge_index, edge_attr, batch):
        h = self.x_embedding1(x[:, 0]) + self.x_embedding2(x[:, 1])
        for i in range(self.num_layer):
            h = self.gnns[i](h, edge_index, edge_attr)
            h = self.batch_norms[i](h)
            if i == self.num_layer - 1:
                h = F.dropout(h, self.drop_ratio, training=self.training)
            else:
                h = F.dropout(F.relu(h), self.drop_ratio, training=self.training)
        h = self.pool(h, batch)
        return self.feat_lin(h)


class CombinedModel(nn.Module):
    def __init__(self, one_model, two_model):
        super().__init__()
        self.one_model = one_model
        self.two_model = two_model

    def forward(self, token_idx, x, edge_index, edge_attr, batch):
        out_1 = self.one_model(token_idx)
        out_2 = self.two_model(x, edge_index, edge_attr, batch)
        return torch.cat((out_1, out_2), dim=1)


class FullModel(nn.Module):
    def __init__(self, combined_model, predictor):
        super().__init__()
        self.combined_model = combined_model
        self.predictor      = predictor

    def forward(self, token_idx, x, edge_index, edge_attr, batch):
        combined_out = self.combined_model(token_idx, x, edge_index, edge_attr, batch)
        return self.predictor(combined_out)


def build_freesolv_predictor(dim):
    # freesolv predictor matches finetune_regression_FreeSolv.py exactly:
    # Linear -> BatchNorm1d -> Dropout(0.2) -> ReLU -> Linear
    return nn.Sequential(
        nn.Linear(dim, dim * 2),
        nn.BatchNorm1d(dim * 2),
        nn.Dropout(p=FREESOLV_PREDICTOR_DROP),
        nn.ReLU(),
        nn.Linear(dim * 2, 1),
    )


def build_esol_predictor(dim):
    # esol predictor matches finetune_regression_ESOL.py exactly:
    # Linear -> Dropout(0.1) -> ReLU -> Linear (no BatchNorm)
    return nn.Sequential(
        nn.Linear(dim, dim * 2),
        nn.Dropout(p=ESOL_PREDICTOR_DROP),
        nn.ReLU(),
        nn.Linear(dim * 2, 1),
    )



_ATOM_LIST = list(range(1, 119))
_CHIRALITY_LIST = [
    Chem.rdchem.ChiralType.CHI_UNSPECIFIED,
    Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CW,
    Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CCW,
    Chem.rdchem.ChiralType.CHI_OTHER,
]
_BOND_LIST    = [BT.SINGLE, BT.DOUBLE, BT.TRIPLE, BT.AROMATIC]
_BONDDIR_LIST = [
    Chem.rdchem.BondDir.NONE,
    Chem.rdchem.BondDir.ENDUPRIGHT,
    Chem.rdchem.BondDir.ENDDOWNRIGHT,
]


@contextlib.contextmanager
def suppress_output():
    # silence stdout and stderr. matches suppress_output() in the dataloaders.
    with open(os.devnull, 'w') as devnull:
        old_stdout, old_stderr = sys.stdout, sys.stderr
        sys.stdout = devnull
        sys.stderr = devnull
        try:
            yield
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr


class MolGraphDataset(Dataset):
    # loads a csv, encodes each smiles through the cfg, and builds rdkit
    # graph features. invalid molecules are filtered during init.
    # matches the MolGraphDataset class in data_process_FreeSolv.py.

    def __init__(self, csv_path, smiles_col, target_col):
        df = pd.read_csv(csv_path)
        self.smiles_raw  = df[smiles_col].tolist()
        self.targets_raw = df[target_col].tolist()
        self.valid_indices = [i for i in range(len(self.smiles_raw)) if self._is_valid(i)]
        self.smiles  = [self.smiles_raw[i]  for i in self.valid_indices]
        self.targets = [self.targets_raw[i] for i in self.valid_indices]
        print(f'  loaded {len(self.smiles)} / {len(self.smiles_raw)} valid molecules from {csv_path}')

    def _is_valid(self, index):
        try:
            smiles = self.smiles_raw[index]
            with suppress_output():
                token_seq = encode_smiles(smiles)
                construct_token_sequence(token_seq, max_len=500)
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                return False
            Chem.AddHs(mol)
            return True
        except Exception:
            return False

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, index):
        smiles = self.smiles[index]
        target = self.targets[index]

        # grammar encoding for the transformer branch.
        with suppress_output():
            token_seq = encode_smiles(smiles)
        tokens_idx, atom_mask = construct_token_sequence(token_seq, max_len=500)
        tokens_idx = np.array(tokens_idx)
        atom_mask  = np.array(atom_mask)
        target     = np.array(target)

        # rdkit graph features for the gcn branch.
        # explicit hydrogens are added as nodes, matching the original dataloaders.
        mol = Chem.MolFromSmiles(smiles)
        mol = Chem.AddHs(mol)

        type_idx, chirality_idx = [], []
        for atom in mol.GetAtoms():
            try:    type_idx.append(_ATOM_LIST.index(atom.GetAtomicNum()))
            except: type_idx.append(0)
            try:    chirality_idx.append(_CHIRALITY_LIST.index(atom.GetChiralTag()))
            except: chirality_idx.append(0)

        x = torch.cat([
            torch.tensor(type_idx,      dtype=torch.long).view(-1, 1),
            torch.tensor(chirality_idx, dtype=torch.long).view(-1, 1),
        ], dim=-1)

        row, col, edge_feat = [], [], []
        for bond in mol.GetBonds():
            start, end = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
            row += [start, end]
            col += [end, start]
            try:    bt = _BOND_LIST.index(bond.GetBondType())
            except: bt = 0
            try:    bd = _BONDDIR_LIST.index(bond.GetBondDir())
            except: bd = 0
            edge_feat += [[bt, bd], [bt, bd]]

        if not row:
            row, col   = [0], [0]
            edge_feat  = [[4, 0]]

        edge_index = torch.tensor([row, col], dtype=torch.long)
        edge_attr  = torch.tensor(edge_feat,  dtype=torch.long)

        return tokens_idx, atom_mask, target, x, edge_index, edge_attr


def molgraph_collate_fn(data):
    # collate function. matches molgraph_collate_fn in the original dataloaders.

    batch_size = len(data)
    tokens_idx_batch = torch.zeros(batch_size, 501, dtype=torch.long)
    atom_mask_batch  = torch.zeros(batch_size, 501, dtype=torch.long)
    target_batch     = torch.zeros(batch_size, 1)

    x_list, edge_index_list, edge_attr_list, batch_indices = [], [], [], []
    num_nodes = 0

    for i in range(batch_size):
        tokens_idx, atom_mask, target, x, edge_index, edge_attr = data[i]
        tokens_idx = torch.tensor(tokens_idx, dtype=torch.long)
        atom_mask  = torch.tensor(atom_mask,  dtype=torch.long)
        target     = torch.tensor(target)

        tokens_idx_batch[i] = tokens_idx
        atom_mask_batch[i]  = atom_mask
        target_batch[i]     = target

        x_list.append(x)
        edge_index_list.append(edge_index + num_nodes)
        edge_attr_list.append(edge_attr)
        batch_indices.append(torch.full((x.size(0),), i, dtype=torch.long))
        num_nodes += x.size(0)

    return (
        tokens_idx_batch,
        atom_mask_batch,
        target_batch,
        torch.cat(x_list,          dim=0),
        torch.cat(edge_index_list,  dim=1),
        torch.cat(edge_attr_list,   dim=0),
        torch.cat(batch_indices,    dim=0),
    )




def compute_splits(length, batch_size):
    # reproduces the exact split logic from the original finetune scripts.
    # returns (train_size, val_size, test_size).

    train = int(length * 0.8)
    test_guodu = length - train

    if test_guodu % batch_size != 0:
        test_guodu = (test_guodu // batch_size) * batch_size
        train = length - test_guodu
        if train % batch_size == 1:
            train += 2
            test_guodu -= 2

    val  = int(test_guodu / 2)
    test = test_guodu - val
    return train, val, test



def run_eval(model, loader, device):
    model.eval()
    total_loss    = 0.0
    total_samples = 0

    with torch.no_grad():
        for token_idx, atom_mask, target, x, edge_index, edge_attr, batch in loader:
            token_idx  = token_idx.to(device)
            atom_mask  = atom_mask.to(device)
            target     = target.to(device)
            x          = x.to(device)
            edge_index = edge_index.to(device)
            edge_attr  = edge_attr.to(device)
            batch      = batch.to(device)

            out = model(token_idx, x, edge_index, edge_attr, batch)

            if torch.isnan(out).any():
                continue

            # sum-reduction mse, then we accumulate and divide at the end.
            # this matches: loss = mse_loss(out, target, reduction='sum').item()
            loss = F.mse_loss(out, target, reduction='sum').item()
            total_loss    += loss
            total_samples += target.numel()

    if total_samples == 0:
        return float('nan')

    mse  = total_loss / total_samples
    rmse = np.sqrt(mse)
    return rmse



def run_dataset(name, csv_path, smiles_col, target_col,
                epochs, lr, batch_size, weight_decay,
                seed, early_stop_patience,
                lr_factor, lr_patience, lr_min,
                gcn_drop, predictor_fn, device):
    # runs the full training and evaluation pipeline for one dataset.
    # all parameters are passed explicitly so both datasets can be run
    # with their original settings from the same script.
    #
    # returns a dict with the benchmark results.

    print(f'\n{"=" * 60}')
    print(f'dataset: {name}')
    print(f'{"=" * 60}')

    # set seeds. matches the exact seed values from the original scripts.
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False

    # load and encode the dataset.
    print('loading and encoding dataset (nltk cfg parsing)...')
    t_load = time.time()
    full_dataset = MolGraphDataset(csv_path, smiles_col, target_col)
    print(f'  dataset ready in {time.time() - t_load:.1f}s')

    length = len(full_dataset)
    indices = list(range(length))

    # split using the exact logic from the original finetune scripts.
    train_n, val_n, test_n = compute_splits(length, batch_size)
    print(f'  split: train={train_n}, val={val_n}, test={test_n}')

    generator = torch.Generator().manual_seed(seed)
    train_indices, val_indices, test_indices = random_split(
        indices, lengths=[train_n, val_n, test_n], generator=generator
    )

    train_dataset = Subset(full_dataset, train_indices)
    val_dataset   = Subset(full_dataset, val_indices)
    test_dataset  = Subset(full_dataset, test_indices)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              collate_fn=molgraph_collate_fn, drop_last=False)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size,
                              collate_fn=molgraph_collate_fn, drop_last=False)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size,
                              collate_fn=molgraph_collate_fn, drop_last=False)

    # build model. architecture matches the original finetune scripts exactly.
    one_model = BERT_atom_embedding_generator(
        d_model=D_MODEL, n_layers=N_LAYERS, vocab_size=VOCAB_SIZE,
        maxlen=MAXLEN, d_k=D_K, d_v=D_V, n_heads=N_HEADS, d_ff=D_FF,
        global_label_dim=1, atom_label_dim=15,
    )
    two_model = GCN(
        num_layer=GCN_LAYERS, emb_dim=GCN_EMB_DIM,
        feat_dim=GCN_FEAT_DIM, drop_ratio=gcn_drop, pool='mean',
    )
    combined_model = CombinedModel(one_model, two_model)
    predictor      = predictor_fn(COMBINED_DIM)
    model          = FullModel(combined_model, predictor).to(device)

    criterion = nn.MSELoss()

    # optimizer. freesolv uses weight_decay, esol does not.
    if weight_decay > 0:
        optimizer = optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.98),
                               weight_decay=weight_decay)
    else:
        optimizer = optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.98))

    # lr scheduler. freesolv uses verbose=True in original; omitted here since
    # we log every epoch anyway.
    lr_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 'min', factor=lr_factor, patience=lr_patience, min_lr=lr_min
    )

    best_metric     = 1e9
    best_epoch      = 0
    best_model      = None
    early_stop      = 0
    test_rmse_list  = []
    t0              = time.time()

    print(f'training for up to {epochs} epochs (early stop patience={early_stop_patience})...')
    print()

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        t1         = time.time()

        for token_idx, atom_mask, target, x, edge_index, edge_attr, batch in train_loader:
            token_idx  = token_idx.to(device)
            atom_mask  = atom_mask.to(device)
            target     = target.to(device)
            x          = x.to(device)
            edge_index = edge_index.to(device)
            edge_attr  = edge_attr.to(device)
            batch      = batch.to(device)

            optimizer.zero_grad()
            out = model(token_idx, x, edge_index, edge_attr, batch)

            if torch.isnan(out).any():
                continue

            loss_batch = criterion(out, target.float())
            train_loss += loss_batch.item() / len(train_loader)
            loss_batch.backward()
            # gradient clipping matches case_MolGramTreeNet.py line 216.
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        val_rmse  = run_eval(model, val_loader,  device)
        test_rmse = run_eval(model, test_loader, device)
        test_rmse_list.append(test_rmse)

        lr_scheduler.step(val_rmse)

        # read the electron_bias value. this is the key diagnostic.
        # we expect it to converge to a positive value for electron-sensitive
        # targets like freesolv and esol.
        bias_val = model.combined_model.one_model.embedding.electron_bias.item()

        print(
            f'  epoch {epoch+1:03d}/{epochs} | '
            f'loss: {train_loss*1e4:.4f} | '
            f'val_rmse: {val_rmse:.4f} | '
            f'test_rmse: {test_rmse:.4f} | '
            f'lr: {optimizer.param_groups[0]["lr"]:.2e} | '
            f'e_bias: {bias_val:.5f} | '
            f't: {time.time()-t1:.1f}s'
        )

        if val_rmse < best_metric:
            best_metric = val_rmse
            best_model  = copy.deepcopy(model)
            best_epoch  = epoch + 1
            early_stop  = 0
        else:
            early_stop += 1

        if early_stop >= early_stop_patience:
            print(f'  early stopping: no improvement for {early_stop_patience} epochs.')
            break

    elapsed = time.time() - t0

    # reload best checkpoint and compute final test rmse.
    best_model.eval()
    final_test_rmse = run_eval(best_model, test_loader, device)
    best_test_rmse  = min(test_rmse_list)
    learned_bias    = best_model.combined_model.one_model.embedding.electron_bias.item()

    print()
    print(f'  total training time : {elapsed/60:.1f} minutes')
    print(f'  best epoch          : {best_epoch}')
    print(f'  best val rmse       : {best_metric:.5f}')
    print(f'  final test rmse     : {final_test_rmse:.5f}')
    print(f'  best test rmse      : {best_test_rmse:.5f}')
    print(f'  learned e_bias      : {learned_bias:.6f}')

    return {
        'dataset':          name,
        'best_epoch':       best_epoch,
        'best_val_rmse':    best_metric,
        'final_test_rmse':  final_test_rmse,
        'best_test_rmse':   best_test_rmse,
        'learned_bias':     learned_bias,
        'training_min':     elapsed / 60,
        'converged_early':  early_stop >= early_stop_patience,
    }




def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'device: {device}')
    print()
    print('model variant        : property-aware molgramtreenet (electron-bias-v1)')
    print(f'transformer          : d_model={D_MODEL}, layers={N_LAYERS}, heads={N_HEADS}')
    print(f'gcn                  : layers={GCN_LAYERS}, emb={GCN_EMB_DIM}, feat={GCN_FEAT_DIM}')
    print(f'electron_bias init   : 0.1')
    print(f'electron token ids   : {sorted(ELECTRON_RELEVANT_TOKEN_SET)}')
    print()

    results = []

    # -- run freesolv --
    # all parameters match finetune_regression_FreeSolv.py exactly.
    r_freesolv = run_dataset(
        name                = 'freesolv',
        csv_path            = FREESOLV_CSV,
        smiles_col          = 'smiles',
        target_col          = 'freesolv',
        epochs              = FREESOLV_EPOCHS,
        lr                  = FREESOLV_LR,
        batch_size          = FREESOLV_BS,
        weight_decay        = FREESOLV_WEIGHT_DECAY,
        seed                = FREESOLV_SEED,
        early_stop_patience = FREESOLV_EARLY_STOP,
        lr_factor           = FREESOLV_LR_FACTOR,
        lr_patience         = FREESOLV_LR_PATIENCE,
        lr_min              = FREESOLV_LR_MIN,
        gcn_drop            = 0.15,
        predictor_fn        = build_freesolv_predictor,
        device              = device,
    )
    results.append(r_freesolv)

    # -- run esol --
    # all parameters match finetune_regression_ESOL.py exactly.
    r_esol = run_dataset(
        name                = 'esol',
        csv_path            = ESOL_CSV,
        smiles_col          = 'smiles',
        target_col          = 'measured log solubility in mols per litre',
        epochs              = ESOL_EPOCHS,
        lr                  = ESOL_LR,
        batch_size          = ESOL_BS,
        weight_decay        = ESOL_WEIGHT_DECAY,
        seed                = ESOL_SEED,
        early_stop_patience = ESOL_EARLY_STOP,
        lr_factor           = ESOL_LR_FACTOR,
        lr_patience         = ESOL_LR_PATIENCE,
        lr_min              = ESOL_LR_MIN,
        gcn_drop            = 0.1,
        predictor_fn        = build_esol_predictor,
        device              = device,
    )
    results.append(r_esol)

    # -- print summary table --
    print()
    print('=' * 72)
    print('benchmark results — property-aware molgramtreenet')
    print('hyperparameters identical to original finetune scripts')
    print('=' * 72)
    print(f'  {"dataset":<12} {"val rmse":>10} {"test rmse":>10} '
          f'{"best epoch":>12} {"e_bias":>10}')
    print(f'  {"-"*12} {"-"*10} {"-"*10} {"-"*12} {"-"*10}')
    for r in results:
        print(
            f'  {r["dataset"]:<12} '
            f'{r["best_val_rmse"]:>10.5f} '
            f'{r["final_test_rmse"]:>10.5f} '
            f'{r["best_epoch"]:>12} '
            f'{r["learned_bias"]:>10.6f}'
        )
    print('=' * 72)

    # -- save csv --
    pd.DataFrame(results).to_csv('benchmark_results.csv', index=False)
    print('results saved to benchmark_results.csv')


if __name__ == '__main__':
    main()


installing dependencies...
  using pyg wheel index: https://data.pyg.org/whl/torch-2.10.0+cu128.html
  installing rdkit...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 50.0 MB/s eta 0:00:00
  nltk: already installed, skipping.
  six: already installed, skipping.
all dependencies installed.
device: cuda

model variant        : property-aware molgramtreenet (electron-bias-v1)
transformer          : d_model=768, layers=6, heads=12
gcn                  : layers=5, emb=300, feat=300
electron_bias init   : 0.1
electron token ids   : [7, 8, 9, 15, 16, 17, 18, 57, 58, 78]


dataset: freesolv
loading and encoding dataset (nltk cfg parsing)...
  loaded 544 / 642 valid molecules from /kaggle/input/datasets/kaalmurlidhar/esolandfreesol/freesolv.csv
  dataset ready in 3.1s
  split: train=448, val=48, test=48
training for up to 200 epochs (early stop patience=30)...

  epoch 001/200 | loss: 146336.7079 | val_rmse: 2.3992 | test_rmse: 2.2143 | lr: 5.00e-05 | e_bias: 0.09862 | t: 22.4s
  e